# Week 6 Assignment - Spark

In [95]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, avg, count, sum

import os

os.environ['SPARK_HOME'] = r"C:\Program Files\spark\spark-4.1.2-bin-hadoop3"
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

spark = SparkSession.builder.appName("Week6Assignment").getOrCreate()
spark

# 1 .Understanding Apache Spark Architecture and Execution Modes

Spark is an open-source distributed processing framework used to deal with big data sets effectively. It uses the master-slave architecture where different processes work together to process programs in parallel.


## Architecture of Apache Spark

Apache Spark architecture consists of the following three main components:

1. Driver

Driver is a program in charge of controlling the execution of the Spark Application. It creates execution plan (Directed Acyclic Graph, DAG), breaks down jobs into tasks, interacts with Cluster Manager and collects the final results.

2. Cluster Manager

Cluster Manager is responsible for resource management within Spark Applications such as CPU and Memory allocation. It manages the cluster and launches Executors on worker nodes. There are different types of cluster managers used in Spark – Standalone, Hadoop YARN, Apache Mesos, Kubernetes.

3. Executors

Executors are worker processes executing the tasks defined by Driver. They run computations in parallel, cache intermediate data, if needed and send results back to Driver.

### Workflow of Spark Architecture Operation

Spark Application execution takes place through the following steps:

- A Spark application is submitted.
- Driver creates execution plan.
- Cluster Manager allocates necessary resources.
- Executors run tasks in parallel.
- Results are sent back to Driver.


## Execution Modes in Spark

The following three modes are offered in Spark:

1) Local Mode

Local Mode involves running of both Driver and Executors on one machine. The use case of this mode is in development, testing, and educational purposes. This mode will be used for the current internship assignment in the VS Code and Jupyter Notebook environment.

2) Client Mode

This mode implies that Driver will run on a local machine, while Executors will be distributed on a cluster. This mode is beneficial for the development process, as logs are easily available on the client machine.

3) Cluster Mode

This mode implies running of both Driver and Executors in a cluster.

## 2. Reading Files with Proper Schema Handling
There are two ways to handle schema when reading a csv file:
1. **InferSchema**:  Spark automatically scans the data to detect data types.
2. **Explicit Schema** : We define our own data types in the schema by using 'StructType'

In [96]:
# 1 InferSchema
csv_file_path = "./data/products.csv"
df_infer = spark.read.csv(csv_file_path,header=True, inferSchema=True)
df_infer.printSchema()
df_infer.show(10)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- user_id: integer (nullable = true)

+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|
|         2| Highlighter| Stationery|  North|    High|  119|   NULL|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|
|         4|       Bench|  Furniture|  North|  Medium|27496|   NULL|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|         7|         Pen| Stationery|Central|  Medium|  298|   NULL|
|         8|       St

In [97]:
# 2 Explicit Schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField(name="product_id",dataType=IntegerType(),nullable=True),
    StructField(name="product_name",dataType=StringType(),nullable=True),
    StructField(name="category",dataType=StringType(),nullable=True),
    StructField(name="region",dataType=StringType(),nullable=True),
    StructField(name="priority",dataType=StringType(),nullable=True),
    StructField(name="price",dataType=StringType(),nullable=True),
    StructField(name="user_id",dataType=StringType(),nullable=True),    
])

df_explicit = spark.read.csv(csv_file_path,header=True,schema=schema)
df_explicit.printSchema()
df_explicit.show(10)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|
|         2| Highlighter| Stationery|  North|    High|  119|   NULL|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|
|         4|       Bench|  Furniture|  North|  Medium|27496|   NULL|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|         7|         Pen| Stationery|Central|  Medium|  298|   NULL|
|         8|       Stoo

# Understanding Lazy Evaluation and DAG (Lineage Graph) in Apache Spark

## Lazy Evaluation
Lazy Evaluation means Spark does not execute transformations such as filter(), select(), or groupBy() immediately. Instead, it stores these operations and executes them only when an action like show(), count(), collect(), or write() is performed.

## DAG(Direct Acyclic Graph)
Directed Acyclic Graph (DAG) forms the execution plan that is formulated by Spark based on the entire set of transformations captured. “Directed” means that the processes follow a specific order, while “Acyclic” refers to the fact that there are no cycles involved. The DAG helps Spark optimize its execution process before data processing.

## Lineage Graph
Lineage Graph consists of all the operations performed on a particular dataset. In case there is any failure of partition, it is possible for Spark to calculate the missing portion only without redoing the entire calculation.

In [98]:
df = spark.read.csv(csv_file_path,header=True,inferSchema=True)
df.show(10)

+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|
|         2| Highlighter| Stationery|  North|    High|  119|   NULL|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|
|         4|       Bench|  Furniture|  North|  Medium|27496|   NULL|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|         7|         Pen| Stationery|Central|  Medium|  298|   NULL|
|         8|       Stool|  Furniture|Central|     Low|22087|   8424|
|         9|      Folder| Stationery|  South|  Medium|  367|   2341|
|        10|         Net|     Sports|Central|     Low| 5777|   5717|
+----------+------------+-----------+-------+--------+-----+-------+
only showing top 10 rows


In [99]:
lazy_plan = df.filter(col("category") == "Electronics").select("product_name","price")
#Here transformation is only defined it does not triggers the job

In [100]:
lazy_plan.explain()
# .explain() shows a physical plan (DAG) without executing it!

== Physical Plan ==
*(1) Project [product_name#2701, price#2705]
+- *(1) Filter (isnotnull(category#2702) AND (category#2702 = Electronics))
   +- FileScan csv [product_name#2701,category#2702,price#2705] Batched: false, DataFilters: [isnotnull(category#2702), (category#2702 = Electronics)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/sachd/Desktop/Celebal Assignments/Week 6 Assignment/dat..., PartitionFilters: [], PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)], ReadSchema: struct<product_name:string,category:string,price:int>




In [101]:
lazy_plan.show()
# Here the action the triggered!

+------------+-----+
|product_name|price|
+------------+-----+
|     Charger|24888|
|       Mouse|10027|
|     Monitor|35902|
|     Speaker|13128|
|     Speaker|26733|
|      Camera|32727|
|       Mouse|67463|
|  Headphones|21443|
|      Tablet| 3570|
|       Phone|24996|
|      Camera|25231|
|      Camera|45653|
|       Phone| 8461|
|      Tablet|37759|
|     Charger|41100|
|       Mouse|14244|
|     Monitor|48327|
|     Monitor|60282|
|    Keyboard|19362|
|     Monitor|66815|
+------------+-----+
only showing top 20 rows


# 4. Performing Filtering and Selection on required columns

In [102]:
df.filter(col("category" ) == "Electronics").select("product_id","product_name","price","category").show()

+----------+------------+-----+-----------+
|product_id|product_name|price|   category|
+----------+------------+-----+-----------+
|         6|     Charger|24888|Electronics|
|        23|       Mouse|10027|Electronics|
|        28|     Monitor|35902|Electronics|
|        30|     Speaker|13128|Electronics|
|        42|     Speaker|26733|Electronics|
|        43|      Camera|32727|Electronics|
|        50|       Mouse|67463|Electronics|
|        54|  Headphones|21443|Electronics|
|        58|      Tablet| 3570|Electronics|
|        60|       Phone|24996|Electronics|
|        62|      Camera|25231|Electronics|
|        64|      Camera|45653|Electronics|
|        72|       Phone| 8461|Electronics|
|        76|      Tablet|37759|Electronics|
|        88|     Charger|41100|Electronics|
|        89|       Mouse|14244|Electronics|
|        93|     Monitor|48327|Electronics|
|        95|     Monitor|60282|Electronics|
|       102|    Keyboard|19362|Electronics|
|       105|     Monitor|66815|E

In [103]:
df.filter((col("region") == "South") & (col("priority") == "High"))\
.select("product_id","product_name","price","region","priority")\
.show(10)

+----------+-------------+-----+------+--------+
|product_id| product_name|price|region|priority|
+----------+-------------+-----+------+--------+
|        29|          Bat| 8804| South|    High|
|        55|         Lamp|15890| South|    High|
|        56|        Scarf| 2032| South|    High|
|        69|Skipping Rope| 1891| South|    High|
|        81|         Belt| 4331| South|    High|
|       133|     Keyboard|54254| South|    High|
|       149|      Frisbee| 6248| South|    High|
|       166|      Monitor|26443| South|    High|
|       170|Skipping Rope| 6709| South|    High|
|       189|        Phone|36558| South|    High|
+----------+-------------+-----+------+--------+
only showing top 10 rows


In [104]:
df.filter((col("price")> 10000)& (col("region")=="North"))\
.select("product_id","product_name","price","region","category")\
.show()

+----------+------------+-----+------+-----------+
|product_id|product_name|price|region|   category|
+----------+------------+-----+------+-----------+
|         4|       Bench|27496| North|  Furniture|
|        19|        Sofa|16701| North|  Furniture|
|        25|    Wardrobe|41721| North|  Furniture|
|        28|     Monitor|35902| North|Electronics|
|        62|      Camera|25231| North|Electronics|
|        64|      Camera|45653| North|Electronics|
|        66|        Lamp|21946| North|  Furniture|
|        89|       Mouse|14244| North|Electronics|
|        95|     Monitor|60282| North|Electronics|
|       107|   Bookshelf|37929| North|  Furniture|
|       117|       Phone|77600| North|Electronics|
|       118|       Bench|21260| North|  Furniture|
|       130|      Camera|17414| North|Electronics|
|       137|  Headphones|62245| North|Electronics|
|       147|       Mouse|63428| North|Electronics|
|       152|        Lamp|44206| North|  Furniture|
|       172|     Monitor|42320|

# 5. Modify DataFrames (rename columns, cast data types, add new columns).

In [105]:
# Renaming Columns
print("Before Renaming")
df.printSchema()

df_revised = (df.withColumnRenamed('product_id','item_id')
              .withColumnRenamed("product_name","item_name"))


print("After Renaming:")
df_revised.printSchema()

Before Renaming
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- user_id: integer (nullable = true)

After Renaming:
root
 |-- item_id: integer (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- user_id: integer (nullable = true)



In [106]:
# Casting Data types
df_revised = df.withColumn("price",col("price").cast("double"))
df_revised.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)



In [107]:
# Adding a new Column
df_revised = df.withColumn("Checkout Price",round(col("price")*1.18))
df_revised.printSchema()
df_revised.show(10)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- Checkout Price: double (nullable = true)

+----------+------------+-----------+-------+--------+-----+-------+--------------+
|product_id|product_name|   category| region|priority|price|user_id|Checkout Price|
+----------+------------+-----------+-------+--------+-----+-------+--------------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|          55.0|
|         2| Highlighter| Stationery|  North|    High|  119|   NULL|         140.0|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|        5563.0|
|         4|       Bench|  Furniture|  North|  Medium|27496|   NULL|       32445.0|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|       24

# 6. Apply transformations and Actions.

In [108]:
# Transformations 
df.show(10)
Stationery = df.filter((col("category") == "Stationery"))  
Columns = df.select("product_id","product_name","region","category")
category_grouped_data = df.groupBy("category").agg(round(avg("price")))

+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|
|         2| Highlighter| Stationery|  North|    High|  119|   NULL|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|
|         4|       Bench|  Furniture|  North|  Medium|27496|   NULL|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|         7|         Pen| Stationery|Central|  Medium|  298|   NULL|
|         8|       Stool|  Furniture|Central|     Low|22087|   8424|
|         9|      Folder| Stationery|  South|  Medium|  367|   2341|
|        10|         Net|     Sports|Central|     Low| 5777|   5717|
+----------+------------+-----------+-------+--------+-----+-------+
only showing top 10 rows


In [109]:
# Actions
Stationery.show(5)
Columns.show(5)
category_grouped_data.show()


+----------+------------+----------+-------+--------+-----+-------+
|product_id|product_name|  category| region|priority|price|user_id|
+----------+------------+----------+-------+--------+-----+-------+
|         1|     Stapler|Stationery|   West|  Medium|   47|   2542|
|         2| Highlighter|Stationery|  North|    High|  119|   NULL|
|         7|         Pen|Stationery|Central|  Medium|  298|   NULL|
|         9|      Folder|Stationery|  South|  Medium|  367|   2341|
|        13|        Glue|Stationery|Central|     Low|  418|   2533|
+----------+------------+----------+-------+--------+-----+-------+
only showing top 5 rows
+----------+------------+-------+----------+
|product_id|product_name| region|  category|
+----------+------------+-------+----------+
|         1|     Stapler|   West|Stationery|
|         2| Highlighter|  North|Stationery|
|         3|     Sweater|  North|  Clothing|
|         4|       Bench|  North| Furniture|
|         5|        Desk|Central| Furniture|
+---

# 7. Understand wide transformations and performance concepts (Shuffle, Predicate Pushdown)

In [110]:
agg = (df.withColumn("price", col("price").cast("double"))
        .groupBy("category")
       .agg(count("*").alias("n"), avg("price").alias("avg_price")))

agg.show()
agg.explain()

+-----------+---+------------------+
|   category|  n|         avg_price|
+-----------+---+------------------+
| Stationery|619| 244.8319870759289|
|     Sports|587| 4509.793867120954|
|Electronics|553| 40262.49367088608|
|   Clothing|642|3071.2429906542056|
|  Furniture|599| 25231.37228714524|
+-----------+---+------------------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#2702], functions=[count(1), avg(price#2943)])
   +- Exchange hashpartitioning(category#2702, 200), ENSURE_REQUIREMENTS, [plan_id=2488]
      +- HashAggregate(keys=[category#2702], functions=[partial_count(1), partial_avg(price#2943)])
         +- Project [category#2702, cast(price#2705 as double) AS price#2943]
            +- FileScan csv [category#2702,price#2705] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/sachd/Desktop/Celebal Assignments/Week 6 Assignment/dat..., PartitionFilters: [], PushedFilters: [], ReadSchema:

In [111]:
# Predicate pushdown: filter on Parquet. Used to look for PushedFilters in the plan.

pushed = spark.read.parquet("data/products_parquet") \
             .filter(col("category") == "Electronics")

pushed.explain()
pushed.show()

== Physical Plan ==
*(1) Filter (isnotnull(category#2979) AND (category#2979 = Electronics))
+- *(1) ColumnarToRow
   +- FileScan parquet [product_id#2977,product_name#2978,category#2979,region#2980,priority#2981,price#2982,user_id#2983] Batched: true, DataFilters: [isnotnull(category#2979), (category#2979 = Electronics)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/sachd/Desktop/Celebal Assignments/Week 6 Assignment/dat..., PartitionFilters: [], PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)], ReadSchema: struct<product_id:int,product_name:string,category:string,region:string,priority:string,price:int...


+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|        23|       Mouse|Electronics|   West|     Low|10027|   8

# 8. CSV vs Parquet
| Aspect | CSV | Parquet |
|---|---|---|
| Layout | Row-based | Columnar |
| Storage | Plain text, uncompressed | Compressed binary |
| Schema | None (infer/define) | Embedded |
| Column pruning | No - reads all columns | Yes - reads only needed columns |
| Predicate pushdown | No | Yes (min/max stats) |


In [112]:
# CSV file reading is shown in the above cells


# Parquet writing 
df.write.mode("overwrite").parquet("data/products_parquet")


In [113]:
df_par = spark.read.parquet("data/products_parquet")
df_par.printSchema()
df_par.show(10)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- user_id: integer (nullable = true)

+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|
|         2| Highlighter| Stationery|  North|    High|  119|   NULL|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|
|         4|       Bench|  Furniture|  North|  Medium|27496|   NULL|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|         7|         Pen| Stationery|Central|  Medium|  298|   NULL|
|         8|       St

In [114]:
# Comparison between CSV and Parquet based on size
def dir_size(path):
    total = 0
    for root,_, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

csv_bytes = os.path.getsize(csv_file_path)
pq_bytes = dir_size("data/products_parquet")
print(f"CSV size: {csv_bytes} bytes")
print(f"Parquet size: {pq_bytes} bytes")
# For large datasets Parquet is typically far smaller and faster to query.

CSV size: 129929 bytes
Parquet size: 43645 bytes


# 9. Handeling Null Values and Filtering Data


In [115]:
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()


+----------+------------+--------+------+--------+-----+-------+
|product_id|product_name|category|region|priority|price|user_id|
+----------+------------+--------+------+--------+-----+-------+
|         0|           0|       0|     0|       0|    0|    226|
+----------+------------+--------+------+--------+-----+-------+



In [116]:
# Removing Null Values
clean = df.filter(col("user_id").isNotNull())
clean.show()

+----------+------------+-----------+-------+--------+-----+-------+
|product_id|product_name|   category| region|priority|price|user_id|
+----------+------------+-----------+-------+--------+-----+-------+
|         1|     Stapler| Stationery|   West|  Medium|   47|   2542|
|         3|     Sweater|   Clothing|  North|     Low| 4714|   3028|
|         5|        Desk|  Furniture|Central|    High|20479|   9858|
|         6|     Charger|Electronics|   East|  Medium|24888|   4078|
|         8|       Stool|  Furniture|Central|     Low|22087|   8424|
|         9|      Folder| Stationery|  South|  Medium|  367|   2341|
|        10|         Net|     Sports|Central|     Low| 5777|   5717|
|        11|        Ball|     Sports|  North|  Medium| 2852|   3490|
|        12|     Sweater|   Clothing|  North|    High|  835|   6140|
|        13|        Glue| Stationery|Central|     Low|  418|   2533|
|        14|      Eraser| Stationery|  North|     Low|  369|   8301|
|        15|    Scissors| Statione

In [117]:
# Verification
clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in clean.columns]).show()

+----------+------------+--------+------+--------+-----+-------+
|product_id|product_name|category|region|priority|price|user_id|
+----------+------------+--------+------+--------+-----+-------+
|         0|           0|       0|     0|       0|    0|      0|
+----------+------------+--------+------+--------+-----+-------+



# 10. Building a Pipeline


In [118]:
src = spark.read.parquet("data/products_parquet")

transformed = (src.withColumn("price",col("price").cast("double"))
               .withColumn("final_price", col("price")*1.18))

final_df = transformed.filter(col("user_id").isNotNull())

final_df.write.mode("overwrite").option("header","true").csv("data/pipeline_output_csv")
final_df.write.mode("overwrite").option("header","true").parquet("data/pipeline_output_parquet")

print("Pipeline Completed")
final_df.show()



Pipeline Completed
+----------+------------+-----------+-------+--------+-------+-------+------------------+
|product_id|product_name|   category| region|priority|  price|user_id|       final_price|
+----------+------------+-----------+-------+--------+-------+-------+------------------+
|         1|     Stapler| Stationery|   West|  Medium|   47.0|   2542|55.459999999999994|
|         3|     Sweater|   Clothing|  North|     Low| 4714.0|   3028|5562.5199999999995|
|         5|        Desk|  Furniture|Central|    High|20479.0|   9858|24165.219999999998|
|         6|     Charger|Electronics|   East|  Medium|24888.0|   4078|          29367.84|
|         8|       Stool|  Furniture|Central|     Low|22087.0|   8424|          26062.66|
|         9|      Folder| Stationery|  South|  Medium|  367.0|   2341|            433.06|
|        10|         Net|     Sports|Central|     Low| 5777.0|   5717|           6816.86|
|        11|        Ball|     Sports|  North|  Medium| 2852.0|   3490|3365.359999

# Summary and Insights

## Summary

In this assignment, I have executed an end-to-end PySpark pipeline for products dataset. First of all, I configured SparkSession locally and read data using both schema inference and StructType. Next, the assignment involved investigation of lazy execution of Spark, filter and select operations, modifications of DataFrame (rename columns, cast type, create new columns), transformation and actions concepts. At the end of the discussion, I have considered performance issues (shuffle, predicate pushdown), differences between CSV and Parquet file formats, treatment of nulls and have built the full read - transform - filter - write pipeline saving results in both formats.

## Insights on Architecture

- Driver vs. Executors: Spark divides planning (Driver) from execution (Executors). Cluster Manager allocates required resources for execution. Driver builds Directed Acyclic Graph (DAG) of transformations and starts dispatching of tasks only when an action is invoked.
- Lazy execution: Transformations (like filter, select, withColumn) build logical plan but do not execute. Catalyst optimizer uses whole DAG to push down filters, prune unnecessary columns and avoid redundant calculations. Method .explain() gives access to this plan before any action execution starts.
- Fault tolerance: DAG provides fault tolerance via lineage mechanism. If some partition is lost, Spark can compute only lost partition reapplying its transformation history to it instead of starting the whole job.

## Insights about Performance 

- Shuffle: Large changes like groupBy trigger a shuffle operation, which transfers data across the network to ensure that the same keys are grouped together. In the physical plan, shuffle operation takes the form of an Exchange operation and is considered the most costly one; therefore, it needs to be avoided as much as possible.
- Predicate pushdown: This is a technique allowing Parquet to skip entire row groups based on embedded min/max statistics and not load data into memory for a filter. In the physical plan, it is represented as PushedFilters. By comparison, CSV is unable to do that because of the absence of embedded statistics.
- CSV vs Parquet: In the case of the current dataset, Parquet was significantly more compact than CSV while having the same data because of its columnar storage format and better compression capabilities. With column pruning and pushdown included, Parquet becomes a better choice for analytics than CSV, which is beneficial for small, human-readable files.
- Practices for working with large datasets: It is recommended to use show(n) or take(n) instead of collect() because it loads all rows into the driver and may lead to out-of-memory errors. Filter early and select only required columns to decrease both shuffle and I/O.

In [119]:
spark.stop()
print("Spark Session has stopped!")

Spark Session has stopped!
